In [ ]:
%%shell
# Ubuntu no longer distributes chromium-browser outside of snap
#
# Proposed solution: https://askubuntu.com/questions/1204571/how-to-install-chromium-without-snap

# Add debian buster
cat > /etc/apt/sources.list.d/debian.list <<'EOF'
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster.gpg] http://deb.debian.org/debian buster main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster-updates.gpg] http://deb.debian.org/debian buster-updates main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-security-buster.gpg] http://deb.debian.org/debian-security buster/updates main
EOF

# Add keys
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A

apt-key export 77E11517 | gpg --dearmour -o /usr/share/keyrings/debian-buster.gpg
apt-key export 22F3D138 | gpg --dearmour -o /usr/share/keyrings/debian-buster-updates.gpg
apt-key export E562B32A | gpg --dearmour -o /usr/share/keyrings/debian-security-buster.gpg

# Prefer debian repo for chromium* packages only
# Note the double-blank lines between entries
cat > /etc/apt/preferences.d/chromium.pref << 'EOF'
Package: *
Pin: release a=eoan
Pin-Priority: 500


Package: *
Pin: origin "deb.debian.org"
Pin-Priority: 300


Package: chromium*
Pin: origin "deb.debian.org"
Pin-Priority: 700
EOF

# Install chromium and chromium-driver
apt-get update
apt-get install chromium chromium-driver

# Install selenium
pip install selenium

Executing: /tmp/apt-key-gpghome.EZOs4f3OkV/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
gpg: key DCC9EFBF77E11517: public key "Debian Stable Release Key (10/buster) <debian-release@lists.debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.QGWciM7osK/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
gpg: key DC30D7C23CBBABEE: public key "Debian Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.glNkscOrKZ/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A
gpg: key 4DFAB270CAA96DFA: public key "Debian Security Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Get:1 http://deb.debian.org/debian buster InRelease [122 kB]
Get:2 http://security.ubuntu.com/ubuntu

In [ ]:
!pip install beautifulsoup4
!pip install lxml
!pip install pandas
!pip install webdriver-manager
!pip install python-dateutil
!pip install bottle
!pip install pivottablejs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 2.5 MB/s eta 0:00:00


In [ ]:
import requests

def download_raids_json(url, save_path):
    try:
        response = requests.get(url)
        response.raise_for_status()  # Check if the request was successful
        with open(save_path, 'wb') as file:
            file.write(response.content)
        print("raids.json downloaded successfully!")
    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch raids.json: {e}")

if __name__ == "__main__":
    url = "https://github.com/bigfoott/ScrapedDuck/raw/data/raids.json"
    save_path = "raids.json"
    download_raids_json(url, save_path)

raids.json downloaded successfully!


In [ ]:
import json

# Path to the existing JSON file
existing_json_file = '/content/raids.json'

# Read the original JSON data from the file
with open(existing_json_file, 'r') as file:
    original_data = json.load(file)

# Create a new JSON object with the "data" wrapper
new_data = {"data": original_data}

# Write the new JSON object back to the file
with open(existing_json_file, 'w') as file:
    json.dump(new_data, file, indent=4)

print("Added 'data' wrapper to the JSON file.")

Added 'data' wrapper to the JSON file.


In [ ]:

import pandas as pd
import requests
import json
from urllib.request import Request,urlopen


pokeId = []
with open('/content/raids.json', 'r') as f:
  data = json.load(f)
for row in data['data']:
  '''nameurl='https://pokeapi.co/api/v2/pokemon/'
  if 'mega' in row['name'].lower():
    megaName = row['name'].lower().split(' ')[1]
    newurl=nameurl+megaName+'-mega'
  else:
    newurl=nameurl+row['name'].lower()
  print(newurl)
  req=Request(
              url=newurl,
              headers={'User-Agent': 'Mozilla/5.0'}
            )
  responseJSON2=urlopen(req)
  dataJSONType=json.loads(responseJSON2.read())
  pokeId.append(dataJSONType['id'])
  print(dataJSONType['id'])'''
  src = row['image']
  if '.icon.' in src:
    src1 = src.split('pm')[1]
    src1 = src1.split('.')[0]
    row['pokeId'] = int(src1)
  else:
    src1 = src.split('pokemon_icon_')[1]
    src1 = src1.split('_')[0]
    row['pokeId'] = int(src1)
with open('/content/raids.json', 'w') as file:
    json.dump(data, file, indent=4)

In [ ]:
import json
import requests
import urllib
from urllib.request import Request,urlopen
from urllib.request import urlopen
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime

from bs4 import BeautifulSoup
from dateutil.relativedelta import relativedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import pandas
import requests
import re
import time
from lxml import html



pokemonNames = []
pokemonId = []
jsondata= []

with open('/content/raids.json', 'r') as f:
  data = json.load(f)

for row in data['data']:
  pokemonNames.append(row['name'])

idurl='https://pokeapi.co/api/v2/pokemon/'

for name in pokemonNames:
  tag=''
  if 'Galarian' in name:
    newName=name.split(' ')
    newName=newName[1]
    tag='-Galarian'
    pokeIdurl=idurl+newName.lower()
    req2=Request(
            url=pokeIdurl,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
    responseJSON2=urlopen(req2)
    dataJSONType=json.loads(responseJSON2.read())
    pokemonId.append(str(dataJSONType['id'])+tag)

  elif 'Alolan' in name:
    newName=name.split(' ')
    newName=newName[1]
    tag='-Alola'
    pokeIdurl=idurl+newName.lower()
    req2=Request(
            url=pokeIdurl,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
    responseJSON2=urlopen(req2)
    dataJSONType=json.loads(responseJSON2.read())
    pokemonId.append(str(dataJSONType['id'])+tag)
  elif 'Mega' in name:
    newName=name.split(' ')
    newName=newName[1]
    tag='-Mega'
    pokeIdurl=idurl+newName.lower()
    req2=Request(
            url=pokeIdurl,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
    responseJSON2=urlopen(req2)
    dataJSONType=json.loads(responseJSON2.read())
    pokemonId.append(str(dataJSONType['id'])+tag)
  elif 'Hisuian' in name:
    newName=name.split(' ')
    newName=newName[1]
    tag='-Hisuian'
    pokeIdurl=idurl+newName.lower()
    req2=Request(
            url=pokeIdurl,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
    responseJSON2=urlopen(req2)
    dataJSONType=json.loads(responseJSON2.read())
    pokemonId.append(str(dataJSONType['id'])+tag)
  elif 'Therian' in name:
    newName=name.split(' ')
    newName=newName[0]
    tag='-incarnate'
    pokeIdurl=idurl+newName.lower()+'-incarnate'
    req2=Request(
            url=pokeIdurl,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
    print(pokeIdurl)
    responseJSON2=urlopen(req2)
    dataJSONType=json.loads(responseJSON2.read())
    pokemonId.append(str(dataJSONType['id'])+tag)

  else:
    pokeIdurl=idurl+name.lower()
    print(name)
    print(pokeIdurl)
    req2=Request(
            url=pokeIdurl,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
    responseJSON2=urlopen(req2)
    dataJSONType=json.loads(responseJSON2.read())
    pokemonId.append(dataJSONType['id'])

url="https://db.pokemongohub.net/pokemon/"

for id in pokemonId:
  print(id)
  if '-Galarian' in str(id):
    counterurl=url+str(id)+'/counters'
    req = Request(
    url=counterurl,
    headers={'User-Agent': 'Mozilla/5.0'}
    )
    webpage = urlopen(req).read()
    soup = BeautifulSoup(webpage, 'html.parser')
    #soup = soup.find_all('div', attrs={'class':'PokemonCounters_results__4xzFH'})
    table = soup.find('table',class_="DataGrid_dataGrid__Q3gQi")
    s=table
    print(s)
    h, [_, *d] = [i.text for i in s.tr.find_all('th')], [[i.text for i in b.find_all('td')] for b in s.find_all('tr')]
    result = [dict(zip(h, i)) for i in d]
    jsondata.append(result)

  elif '-Alola' in str(id):

    counterurl=url+str(id)+'/counters'
    req = Request(
    url=counterurl,
    headers={'User-Agent': 'Mozilla/5.0'}
    )
    webpage = urlopen(req).read()
    soup = BeautifulSoup(webpage, 'html.parser')
    #soup = soup.find_all('div', attrs={'class':'PokemonCounters_results__4xzFH'})
    table = soup.find('table',class_="DataGrid_dataGrid__Q3gQi")
    s=table
    print(s)
    h, [_, *d] = [i.text for i in s.tr.find_all('th')], [[i.text for i in b.find_all('td')] for b in s.find_all('tr')]
    result = [dict(zip(h, i)) for i in d]
    jsondata.append(result)

  elif '-Mega' in str(id):
    counterurl=url+str(id)+'/counters'
    req = Request(
    url=counterurl,
    headers={'User-Agent': 'Mozilla/5.0'}
    )
    webpage = urlopen(req).read()
    soup = BeautifulSoup(webpage, 'html.parser')
    #soup = soup.find_all('div', attrs={'class':'PokemonCounters_results__4xzFH'})
    table = soup.find('table',class_="DataGrid_dataGrid__Q3gQi")
    s=table
    print(s)
    h, [_, *d] = [i.text for i in s.tr.find_all('th')], [[i.text for i in b.find_all('td')] for b in s.find_all('tr')]
    result = [dict(zip(h, i)) for i in d]
    jsondata.append(result)

  elif '-Hisui' in str(id):
    counterurl=url+str(id)+'/counters'
    print(counterurl)
    req = Request(
    url=counterurl,
    headers={'User-Agent': 'Mozilla/5.0'}
    )
    webpage = urlopen(req).read()
    soup = BeautifulSoup(webpage, 'html.parser')
    #soup = soup.find_all('div', attrs={'class':'PokemonCounters_results__4xzFH'})
    table = soup.find('table',class_="DataGrid_dataGrid__Q3gQi")
    s=table
    print(s)
    h, [_, *d] = [i.text for i in s.tr.find_all('th')], [[i.text for i in b.find_all('td')] for b in s.find_all('tr')]
    result = [dict(zip(h, i)) for i in d]
    jsondata.append(result)

  elif '-incarnate' in str(id):
    ounterurl=url+str(id)+'/counters'
    print(counterurl)
    req = Request(
    url=counterurl,
    headers={'User-Agent': 'Mozilla/5.0'}
    )
    webpage = urlopen(req).read()
    soup = BeautifulSoup(webpage, 'html.parser')
    #soup = soup.find_all('div', attrs={'class':'PokemonCounters_results__4xzFH'})
    table = soup.find('table',class_="DataGrid_dataGrid__Q3gQi")
    s=table
    print(s)
    h, [_, *d] = [i.text for i in s.tr.find_all('th')], [[i.text for i in b.find_all('td')] for b in s.find_all('tr')]
    result = [dict(zip(h, i)) for i in d]
    jsondata.append(result)


  else:
    counterurl=url+str(id)+'/counters'
    req = Request(
    url=counterurl,
    headers={'User-Agent': 'Mozilla/5.0'}
    )
    webpage = urlopen(req).read()
    soup = BeautifulSoup(webpage, 'html.parser')
    #soup = soup.find_all('div', attrs={'class':'PokemonCounters_results__4xzFH'})
    table = soup.find('table',class_="DataGrid_dataGrid__Q3gQi")
    s=table
    print(s)
    h, [_, *d] = [i.text for i in s.tr.find_all('th')], [[i.text for i in b.find_all('td')] for b in s.find_all('tr')]
    result = [dict(zip(h, i)) for i in d]
    jsondata.append(result)

#for i in range(len(pokemonNames)):
    #counterjson = '{ "data": [ { "pokemon"} ]  }'
list_of_lists = []

for name, data in zip(pokemonNames, jsondata):
    combined_data = [name, data]
    list_of_lists.append(combined_data)

data = []

for item in list_of_lists:
    pokemon_data = {
        "pokemon": item[0],
        "counter": item[1]
    }
    data.append(pokemon_data)

result = {"data": data}

# Convert the dictionary to JSON format
json_data = json.dumps(result, ensure_ascii=False, indent=4)

with open('./counter.json',"w") as output_file:
    output_file.write(json.dumps({"data": data}, indent=4 ))


Bronzor
https://pokeapi.co/api/v2/pokemon/bronzor
Goomy
https://pokeapi.co/api/v2/pokemon/goomy
Klink
https://pokeapi.co/api/v2/pokemon/klink
Beldum
https://pokeapi.co/api/v2/pokemon/beldum
Magneton
https://pokeapi.co/api/v2/pokemon/magneton
Lucario
https://pokeapi.co/api/v2/pokemon/lucario
Aggron
https://pokeapi.co/api/v2/pokemon/aggron
Registeel
https://pokeapi.co/api/v2/pokemon/registeel
436
<table class="DataGrid_dataGrid__Q3gQi"><thead><tr><th colspan="1"><div class="cursor-pointer select-none">#</div></th><th colspan="1"><div class="cursor-pointer select-none">Name</div></th><th colspan="1"><div class="cursor-pointer select-none">Fast Attack</div></th><th colspan="1"><div class="cursor-pointer select-none">Charged Attack</div></th><th colspan="1"><div class="cursor-pointer select-none">DPS</div></th><th colspan="1"><div class="cursor-pointer select-none">TDO</div></th><th colspan="1"><div class="cursor-pointer select-none">Faints</div></th><th colspan="1"><div class="cursor-point

In [ ]:
def strip_after_parentheses(text):
    index = text.find('(')
    if index != -1:
        return text[:index]
    else:
        return text
def strip_after_plus(text):
    index = text.find('+')
    if index != -1:
        return text[:index]
    else:
        return text
moveurl = 'https://pokeapi.co/api/v2/move/'
for type_data in data:
  for move in type_data['counter']:
    movefast = move['Fast Attack'].replace('*','').strip().replace(' ','-').lower()
    movefast = strip_after_parentheses(movefast)
    movefast = strip_after_plus(movefast)
    chargedmove = move['Charged Attack'].replace('*','').strip().replace(' ','-').lower()
    chargedmove = strip_after_parentheses(chargedmove)
    chargedmove = strip_after_plus(chargedmove)
    fastmoveurl = moveurl + movefast
    chargedmoveurl = moveurl + chargedmove
    print(chargedmoveurl)
    req3=Request(
            url=fastmoveurl,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
    responseJSON3=urlopen(req3)
    dataJSONType3=json.loads(responseJSON3.read())
    req4=Request(
            url=chargedmoveurl,
            headers={'User-Agent': 'Mozilla/5.0'}
        )
    responseJSON4=urlopen(req4)
    dataJSONType4=json.loads(responseJSON4.read())
    move['Fast Attack'] = move['Fast Attack'].replace('*','').strip()+'('+dataJSONType3['type']['name']+')'
    move['Charged Attack'] = move['Charged Attack'].replace('*','').strip()+'('+dataJSONType4['type']['name']+')'


https://pokeapi.co/api/v2/move/precipice-blades
https://pokeapi.co/api/v2/move/blast-burn
https://pokeapi.co/api/v2/move/brutal-swing
https://pokeapi.co/api/v2/move/blast-burn
https://pokeapi.co/api/v2/move/shadow-ball
https://pokeapi.co/api/v2/move/precipice-blades
https://pokeapi.co/api/v2/move/earth-power
https://pokeapi.co/api/v2/move/fusion-flare
https://pokeapi.co/api/v2/move/shadow-ball
https://pokeapi.co/api/v2/move/brutal-swing
https://pokeapi.co/api/v2/move/overheat
https://pokeapi.co/api/v2/move/foul-play
https://pokeapi.co/api/v2/move/overheat
https://pokeapi.co/api/v2/move/overheat
https://pokeapi.co/api/v2/move/scorching-sands
https://pokeapi.co/api/v2/move/blast-burn
https://pokeapi.co/api/v2/move/sacred-fire
https://pokeapi.co/api/v2/move/shadow-ball
https://pokeapi.co/api/v2/move/crunch
https://pokeapi.co/api/v2/move/earth-power
https://pokeapi.co/api/v2/move/magma-storm
https://pokeapi.co/api/v2/move/sandsear-storm
https://pokeapi.co/api/v2/move/blast-burn
https://pok

In [ ]:
with open('./counter.json',"w") as output_file:
    output_file.write(json.dumps({"data": data}, indent=4 ))


In [ ]:
import requests
import json

url = "https://getpantry.cloud/apiv1/pantry/b45d3e57-17a6-498d-8aec-b8173408efb4/basket/counters"

# Load data from raids.json file
with open('/content/counter.json') as f:
    payload = json.load(f)

# Convert the payload to a JSON string
payload_json = json.dumps(payload)

headers = {
    'Content-Type': 'application/json'
}

response = requests.post(url, headers=headers, data=payload_json)

print(response.text)

Your Pantry was updated with basket: counters!
